Альтернативной задачей буду решать детекцию, определяя баунбоксы зданий по картинке. Для обучения воспользуемся исходным датасетом с предобработкой: по бинарной маске зданий определим огибающие прямоугольники и их подадим в обучающую и валидационную выборку.
В качестве начальной модели возьмем готовую модель FasterRCNN_ResNet50 и дообучим ее.
Обучение проведем на 50 эпохах, запоминая модель в случае появления лучшего решения по валидационной выборке.

In [1]:
pip install torch torchvision opencv-python numpy scikit-learn

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms
from sklearn.model_selection import train_test_split
from pathlib import Path
import glob

# ----------------------------
# 1. Вспомогательные функции
# ----------------------------

def mask_to_bboxes(mask_path):
    """Извлекает bounding boxes из бинарной маски."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    bboxes = []
    for i in range(1, num_labels):
        x, y, w, h = stats[i][:4]
        if w >= 10 and h >= 10:  # фильтр шума
            bboxes.append([x, y, x + w, y + h])
    return bboxes

def calculate_iou(box1, box2):
    """Вычисляет IoU между двумя bbox [x1, y1, x2, y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area

    return inter_area / union_area if union_area > 0 else 0.0

def evaluate_metrics(pred_boxes, target_boxes, iou_threshold=0.5):
    """
    Простая метрика: точность (accuracy) и средний IoU.
    Accuracy = доля предсказанных bbox, имеющих match с IoU > threshold.
    """
    if len(pred_boxes) == 0 and len(target_boxes) == 0:
        return 1.0, 1.0
    if len(pred_boxes) == 0 or len(target_boxes) == 0:
        return 0.0, 0.0

    matched = 0
    total_iou = 0.0
    used_targets = set()

    for pred in pred_boxes:
        best_iou = 0
        best_idx = -1
        for i, tgt in enumerate(target_boxes):
            if i in used_targets:
                continue
            iou = calculate_iou(pred, tgt)
            if iou > best_iou:
                best_iou = iou
                best_idx = i
        if best_iou >= iou_threshold:
            matched += 1
            total_iou += best_iou
            used_targets.add(best_idx)

    accuracy = matched / len(pred_boxes) if len(pred_boxes) > 0 else 0.0
    avg_iou = total_iou / matched if matched > 0 else 0.0
    return avg_iou, accuracy

# ----------------------------
# 2. Датасет
# ----------------------------

def mask_to_bboxes_from_mask(mask):
    """Принимает np.ndarray [H, W]"""
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    bboxes = []
    for i in range(1, num_labels):
        x, y, w, h = stats[i][:4]
        if w >= 5 and h >= 5:  
            bboxes.append([x, y, x + w, y + h])
    return bboxes

class HouseDetectionDataset(Dataset):
    def __init__(self, image_paths, mask_paths, max_size=1024):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.max_size = max_size  
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)  

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        # Загрузка
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        # Ресайз с сохранением пропорций
        h, w = image.shape[:2]
        scale = self.max_size / max(h, w)
        if scale < 1.0:
            new_h, new_w = int(h * scale), int(w * scale)
            image = cv2.resize(image, (new_w, new_h))
            mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

        image = self.transform(image)
        
        # Преобразование bbox'ов
        boxes = mask_to_bboxes_from_mask(mask)  
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((len(boxes),), dtype=torch.int64)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd
        }
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

# ----------------------------
# 3. Модель
# ----------------------------

def get_model(num_classes=2):
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# ----------------------------
# 4. Обучение
# ----------------------------

def train_detector():
    # Пути к данным
    image_dir = "Data/train/images"
    mask_dir = "Data/train/masks"

    # Сбор файлов
    image_extensions = {".jpg", ".tif", ".png"}
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(image_dir, f"*{ext}")))
        image_files.extend(glob.glob(os.path.join(image_dir, f"*{ext.upper()}")))
    image_files = sorted(list(set(image_files)))

    image_paths = []
    mask_paths = []
    for img_path in image_files:
        img_name = Path(img_path).stem
        mask_found = False
        for ext in image_extensions:
            mask_path = os.path.join(mask_dir, img_name + ext)
            if os.path.exists(mask_path):
                image_paths.append(img_path)
                mask_paths.append(mask_path)
                mask_found = True
                break
            mask_path = os.path.join(mask_dir, img_name + ext.upper())
            if os.path.exists(mask_path):
                image_paths.append(img_path)
                mask_paths.append(mask_path)
                mask_found = True
                break
        if not mask_found:
            print(f"Пропущено: {img_name}")

    print(f"Найдено {len(image_paths)} пар.")

    # Разделение
    X_train, X_val, y_train, y_val = train_test_split(
        image_paths, mask_paths, test_size=0.2, random_state=42
    )

    # Датасеты
    train_dataset = HouseDetectionDataset(X_train, y_train, max_size=1024)
    val_dataset = HouseDetectionDataset(X_val, y_val, max_size=1024)

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=2)

    # Устройство
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Устройство: {device}")

    # Модель
    model = get_model().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
    best_val_loss = float('inf')

    # Обучение
    num_epochs = 50
    log_every = 20

    for epoch in range(num_epochs):
        print(f"\nЭпоха {epoch+1}/{num_epochs}")
        model.train()
        train_loss = 0.0
        train_iou = 0.0
        train_acc = 0.0
        batch_count = 0
        metric_batches = 0

        for batch_idx, (images, targets) in enumerate(train_loader):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            train_loss += losses.item()
            batch_count += 1

            # Расчёт метрик каждые log_every батчей
            if batch_idx % log_every == 0:
                model.eval()
                with torch.no_grad():
                    pred_dicts = model(images)
                    for pred, target in zip(pred_dicts, targets):
                        pred_boxes = pred['boxes'].cpu().numpy()
                        target_boxes = target['boxes'].cpu().numpy()
                        iou, acc = evaluate_metrics(pred_boxes, target_boxes)
                        train_iou += iou
                        train_acc += acc
                        metric_batches += 1
                model.train()

        avg_train_loss = train_loss / len(train_loader)
        avg_train_iou = train_iou / metric_batches if metric_batches > 0 else 0.0
        avg_train_acc = train_acc / metric_batches if metric_batches > 0 else 0.0

        # Валидация
        model.eval()
        val_loss = 0.0
        val_iou = 0.0
        val_acc = 0.0
        val_batches = 0

        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                #Временно включаем режим обучения, чтобы получить loss
                model.train()
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
                val_loss += losses.item()

                #Снова в eval для получения предсказаний
                model.eval()
                pred_dicts = model(images)

                for pred, target in zip(pred_dicts, targets):
                    pred_boxes = pred['boxes'].cpu().numpy()
                    target_boxes = target['boxes'].cpu().numpy()
                    iou, acc = evaluate_metrics(pred_boxes, target_boxes)
                    val_iou += iou
                    val_acc += acc
                    val_batches += 1

        model.train() 

        avg_val_loss = val_loss / len(val_loader)
        avg_val_iou = val_iou / val_batches if val_batches > 0 else 0.0
        avg_val_acc = val_acc / val_batches if val_batches > 0 else 0.0

        print(f"Train Loss: {avg_train_loss:.4f} | Train IoU: {avg_train_iou:.4f} | Train Acc: {avg_train_acc:.4f}")
        print(f"Val   Loss: {avg_val_loss:.4f} | Val   IoU: {avg_val_iou:.4f} | Val   Acc: {avg_val_acc:.4f}")

        # Сохранение лучшей модели
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "house_detector_best.pth")
            print("Лучшая модель сохранена")

    print("\nОбучение завершено!")

if __name__ == "__main__":
    train_detector()

Найдено 180 пар.
Устройство: cuda


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /tmp/xdg_cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:02<00:00, 71.5MB/s] 



Эпоха 1/50
Train Loss: 1.7737 | Train IoU: 0.6034 | Train Acc: 0.1225
Val   Loss: 1.5062 | Val   IoU: 0.6100 | Val   Acc: 0.1403
Лучшая модель сохранена

Эпоха 2/50
Train Loss: 1.5321 | Train IoU: 0.6241 | Train Acc: 0.2838
Val   Loss: 1.4518 | Val   IoU: 0.6263 | Val   Acc: 0.2789
Лучшая модель сохранена

Эпоха 3/50
Train Loss: 1.4646 | Train IoU: 0.6613 | Train Acc: 0.3663
Val   Loss: 1.3948 | Val   IoU: 0.6435 | Val   Acc: 0.3519
Лучшая модель сохранена

Эпоха 4/50
Train Loss: 1.4335 | Train IoU: 0.6760 | Train Acc: 0.4625
Val   Loss: 1.3793 | Val   IoU: 0.6551 | Val   Acc: 0.4031
Лучшая модель сохранена

Эпоха 5/50
Train Loss: 1.3726 | Train IoU: 0.6583 | Train Acc: 0.5100
Val   Loss: 1.3642 | Val   IoU: 0.6684 | Val   Acc: 0.4164
Лучшая модель сохранена

Эпоха 6/50
Train Loss: 1.3306 | Train IoU: 0.6774 | Train Acc: 0.5700
Val   Loss: 1.3155 | Val   IoU: 0.6654 | Val   Acc: 0.4283
Лучшая модель сохранена

Эпоха 7/50
Train Loss: 1.3159 | Train IoU: 0.6980 | Train Acc: 0.6012
Val  

Лучшая модель получена на 45й эпохе с параметрами:
Train Loss: 0.9298 | Train IoU: 0.7367 | Train Acc: 0.7300
Val   Loss: 1.1585 | Val   IoU: 0.6996 | Val   Acc: 0.6018

Результаты хуже, чем у модели сегментации. Это происходит по причине:
1. Наличия непрямоугольных или повернутых зданий. Модель детекции может не распознать или разпознать, но с большой погрешностью, данные здания.
2. Отсутствия датасета с размеченными баунбоксами. Автоматическая разметка также приведет к погрешности.

Но в целом, несмотря на скептицизм, модель получила  Val   IoU: 0.6996 | Val   Acc: 0.6018 

